# Hands-on 1: Precipitation Forecast Post-Processing

**Module 2 — Deep Learning Best Practices & Applications**  
CIMA Foundation PhD Course, 27 May 2026

In this notebook we build a simple CNN to post-process (bias-correct) NWP precipitation forecasts.  
We use the **Modern ML Stack**: Zarr + xarray for data, PyTorch Lightning for training, Hydra/OmegaConf for configuration, and MLFlow for experiment tracking.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/meteocima/Monaco_DLCourse/blob/main/notebooks/01_precipitation_postprocessing.ipynb)

## 0. Setup

We use [**uv**](https://docs.astral.sh/uv/) as our Python package manager — it's a fast, modern replacement for pip + virtualenv.

| Context | Command | What it does |
|---------|---------|-------------|
| **New project** | `uv init` | Scaffold a project with `pyproject.toml` |
| **Add a dependency** | `uv add torch xarray` | Add packages to `pyproject.toml` and install them |
| **Install everything** | `uv sync` | Create a `.venv` and install from the lockfile |
| **Run a command** | `uv run jupyter lab` | Run inside the managed `.venv` |
| **Colab / system Python** | `uv pip install --system -r requirements.txt` | Install without a `.venv` (for environments you don't manage) |

On **Colab** there is no `.venv` — Google provides the Python environment — so we use `uv pip install --system`.

In [ ]:
import os

# On Colab: clone the repo (or pull latest) and install dependencies
if "COLAB_RELEASE_TAG" in os.environ:
    if os.path.exists("/content/Monaco_DLCourse"):
        !git -C /content/Monaco_DLCourse pull -q
    else:
        !git clone https://github.com/meteocima/Monaco_DLCourse.git /content/Monaco_DLCourse
    %cd /content/Monaco_DLCourse/notebooks
    !pip install -q uv
    !uv pip install --system -q -r ../requirements.txt

In [ ]:
from __future__ import annotations

from typing import Any

import cartopy.crs as ccrs
import cartopy.feature as cfeature
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import xarray as xr
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import pytorch_lightning as pl
from omegaconf import DictConfig, OmegaConf
import mlflow

print(f"PyTorch version: {torch.__version__}")
print(f"Lightning version: {pl.__version__}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print(
        "\n⚠️  No GPU detected!\n"
        "   Go to Runtime → Change runtime type → T4 GPU\n"
        "   Then re-run this cell.\n"
        "   Training will still work on CPU, but will be slower."
    )

## 1. Configuration with Hydra / OmegaConf

In production, Hydra loads YAML config files and composes them automatically via its `@hydra.main` decorator. In a notebook we use **OmegaConf** (Hydra's config backend) to load the same YAML files directly — giving us structured, typed configuration without needing the CLI.

The `load_config()` function below loads the YAML and lets you **override any value** via keyword arguments — the same pattern Hydra uses on the command line (`max_epochs=50`), but adapted for notebooks.

In [ ]:
def load_config(
    config_path: str = "../configs/precipitation.yaml",
    **overrides: Any,
) -> DictConfig:
    """Load a YAML config and apply keyword overrides.

    Args:
        config_path: Path to the YAML configuration file.
        **overrides: Flat shorthand keys (e.g. ``max_epochs=50``,
            ``batch_size=64``) are mapped to their nested config
            paths automatically.

    Returns:
        Merged OmegaConf ``DictConfig``.
    """
    cfg = OmegaConf.load(config_path)

    # Map flat shorthand keys to their nested config paths
    SHORTCUTS: dict[str, str] = {
        "max_lead_time": "data.max_lead_time",
        "max_epochs": "training.max_epochs",
        "hidden_channels": "model.hidden_channels",
        "n_layers": "model.n_layers",
        "learning_rate": "training.learning_rate",
        "batch_size": "data.batch_size",
    }
    for key, value in overrides.items():
        path = SHORTCUTS.get(key, key)
        OmegaConf.update(cfg, path, value)

    return cfg


cfg = load_config()
print(OmegaConf.to_yaml(cfg))

## 2. Data Loading with Zarr + xarray

The `load_data()` function encapsulates the full data pipeline:

1. **Open** the WeatherBench 2 cloud Zarr stores (lazy — no data downloaded yet)
2. **Select** ERA5 truth and HRES forecasts at all 6-hourly lead times up to `max_lead_time`
3. **Convert** from metres to **mm** (× 1000)
4. **Normalise** using min-max scaling with 1st / 99th percentiles (computed on the training split only)

Each (initialisation time, lead time) pair becomes one sample. The **lead time** is encoded as a normalised scalar in [0, 1] so the CNN can learn lead-time-dependent corrections.

In [ ]:
def load_data(cfg: DictConfig) -> dict[str, Any]:
    """Load precipitation forecasts and truth from WeatherBench 2.

    Opens the ERA5 and HRES Zarr stores lazily, selects the configured
    time range and lead-time steps, converts to mm, and applies min-max
    normalisation based on training-split percentiles.

    Args:
        cfg: Configuration with ``data.*`` keys (URLs, time range,
            ``max_lead_time``, ``train_split``, ``lat_slice``).

    Returns:
        Dict with normalised arrays, original-mm arrays, coordinates,
        normalisation statistics, and metadata.
    """
    gcs_opts = {"token": "anon"}
    ds_era5 = xr.open_zarr(
        cfg.data.era5_zarr_url, storage_options=gcs_opts
    )
    ds_hres = xr.open_zarr(
        cfg.data.hres_zarr_url,
        storage_options=gcs_opts,
        decode_timedelta=True,
    )

    print("ERA5 variables:", list(ds_era5.data_vars))
    print("HRES variables:", list(ds_hres.data_vars))
    print(
        f"\nHRES lead times: "
        f"{ds_hres.prediction_timedelta.values[[0, -1]]}"
    )

    max_lead = pd.Timedelta(cfg.data.max_lead_time)
    steps = pd.timedelta_range("6h", max_lead, freq="6h")
    print(f"Lead-time steps ({len(steps)}): {steps[0]}  …  {steps[-1]}")

    fc_raw = (
        ds_hres[cfg.data.variable]
        .sel(prediction_timedelta=steps)
        .sel(time=slice(cfg.data.time_start, cfg.data.time_end))
        .isel(latitude=slice(0, cfg.data.lat_slice))
        .transpose("time", "prediction_timedelta", "latitude", "longitude")
    )
    print(f"HRES shape: {dict(fc_raw.sizes)}")

    init_times = fc_raw.time.values
    n_init = len(init_times)
    n_steps = len(steps)

    # Compute valid times for every (init, step) pair
    valid_times_2d = init_times[:, None] + steps.values[None, :]

    truth_raw = (
        ds_era5[cfg.data.variable]
        .sel(time=valid_times_2d.ravel())
        .isel(latitude=slice(0, cfg.data.lat_slice))
        .transpose("time", "latitude", "longitude")
    )

    lats = fc_raw.latitude.values
    lons = fc_raw.longitude.values

    # WeatherBench stores precipitation in metres — convert to mm
    forecast_mm = (
        fc_raw.values.astype(np.float32).reshape(
            -1, len(lats), len(lons)
        )
        * 1000
    )
    truth_mm = truth_raw.values.astype(np.float32) * 1000

    # Min-max normalisation using 1st / 99th percentiles (training split)
    n_train = int(len(forecast_mm) * cfg.data.train_split)
    train_all = np.concatenate(
        [forecast_mm[:n_train], truth_mm[:n_train]]
    )
    precip_p1 = np.percentile(train_all, 1).astype(np.float32)
    precip_p99 = np.percentile(train_all, 99).astype(np.float32)
    print(
        f"Precip normalisation: p1 = {precip_p1:.4f} mm, "
        f"p99 = {precip_p99:.4f} mm"
    )

    forecast = (forecast_mm - precip_p1) / (precip_p99 - precip_p1)
    truth = (truth_mm - precip_p1) / (precip_p99 - precip_p1)

    step_hours = (steps.total_seconds() / 3600).values.astype(np.float32)
    max_hours = float(max_lead.total_seconds() / 3600)
    lead_norm = np.tile(
        (step_hours / max_hours).astype(np.float32), n_init
    )

    print(f"\nSamples: {len(forecast)}  ({n_init} init × {n_steps} steps)")
    print(
        f"Forecast: {forecast.shape}, Truth: {truth.shape}, "
        f"Lead norm: {lead_norm.shape}"
    )
    print(
        f"Mean bias (forecast − truth): "
        f"{(forecast_mm - truth_mm).mean():.4f} mm"
    )

    return {
        "forecast": forecast,
        "truth": truth,
        "lead_norm": lead_norm,
        "forecast_mm": forecast_mm,
        "truth_mm": truth_mm,
        "lats": lats,
        "lons": lons,
        "precip_p1": precip_p1,
        "precip_p99": precip_p99,
        "precip_scale": precip_p99 - precip_p1,
        "n_init": n_init,
        "n_steps": n_steps,
        "step_hours": step_hours,
        "max_hours": max_hours,
        "init_times": init_times,
    }


data = load_data(cfg)

### Visualising the raw data

Before training, it's important to inspect the data. `plot_raw_data()` shows a single forecast / truth / error triplet on a map, plus the **RMSE vs lead time** curve for the first issue date. This helps you spot systematic biases that the CNN should learn to correct.

In [ ]:
def plot_raw_data(
    data: dict[str, Any],
    sample_idx: int = -1,
) -> None:
    """Visualise a single forecast / truth pair and RMSE vs lead time.

    Args:
        data: Dict returned by ``load_data()``.
        sample_idx: Index into the first initialisation time's lead
            steps to display.  ``-1`` (default) picks the last step.
    """
    forecast = data["forecast"]
    truth = data["truth"]
    lats = data["lats"]
    lons = data["lons"]
    step_hours = data["step_hours"]
    n_steps = data["n_steps"]
    init_times = data["init_times"]
    precip_p1 = data["precip_p1"]
    precip_scale = data["precip_scale"]

    def _to_mm(x: np.ndarray) -> np.ndarray:
        return x * precip_scale + precip_p1

    if sample_idx < 0:
        sample_idx = n_steps + sample_idx
    idx = sample_idx

    sample_lead_h = step_hours[idx % n_steps]
    issue_date = pd.Timestamp(init_times[0]).strftime("%Y-%m-%d %HZ")
    proj = ccrs.PlateCarree()
    lon2d, lat2d = np.meshgrid(lons, lats)

    fc_mm = _to_mm(forecast[idx])
    tr_mm = _to_mm(truth[idx])

    fig, axes = plt.subplots(
        1, 3, figsize=(18, 5), subplot_kw={"projection": proj}
    )
    titles = ["IFS HRES Forecast", "ERA5 Truth", "Forecast Error"]
    plot_data = [fc_mm, tr_mm, fc_mm - tr_mm]
    cmaps = ["Blues", "Blues", "RdBu_r"]
    vmin_max = [(0, None), (0, None), (None, None)]

    for ax, title, d, cmap, (vn, vx) in zip(
        axes, titles, plot_data, cmaps, vmin_max
    ):
        im = ax.pcolormesh(
            lon2d, lat2d, d, cmap=cmap, vmin=vn, vmax=vx,
            transform=proj,
        )
        ax.add_feature(cfeature.COASTLINE, linewidth=0.5)
        ax.add_feature(cfeature.BORDERS, linewidth=0.3, linestyle="--")
        ax.gridlines(draw_labels=True, linewidth=0.3, alpha=0.5)
        ax.set_title(title)
        plt.colorbar(
            im, ax=ax, orientation="horizontal", pad=0.05, shrink=0.8
        )

    fig.suptitle(
        f"6h Precipitation — {sample_lead_h:.0f}h lead, "
        f"issued {issue_date} (mm)",
        fontsize=14, y=1.02,
    )
    plt.tight_layout()
    plt.show()

    # RMSE vs lead time for the first issue date
    rmse_per_step = np.array([
        np.sqrt(np.mean((_to_mm(forecast[i]) - _to_mm(truth[i])) ** 2))
        for i in range(n_steps)
    ])

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(step_hours, rmse_per_step, "o-", color="tab:blue")
    ax.set_xlabel("Lead time (hours)")
    ax.set_ylabel("RMSE (mm)")
    ax.set_title(
        f"HRES vs ERA5 — RMSE by lead time (issued {issue_date})"
    )
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


plot_raw_data(data)

## 3. PyTorch Dataset & DataModule

In [ ]:
class PrecipitationDataset(Dataset):
    """Dataset for precipitation post-processing with lead-time channel."""

    def __init__(
        self,
        forecast: np.ndarray,
        truth: np.ndarray,
        lead_norm: np.ndarray,
    ) -> None:
        self.forecast = torch.from_numpy(forecast).unsqueeze(1)  # (N,1,H,W)
        self.truth = torch.from_numpy(truth).unsqueeze(1)
        self.lead_norm = torch.from_numpy(lead_norm).float()     # (N,)

    def __len__(self) -> int:
        return len(self.forecast)

    def __getitem__(
        self, idx: int
    ) -> tuple[torch.Tensor, torch.Tensor]:
        fc = self.forecast[idx]                         # (1, H, W)
        lead_ch = self.lead_norm[idx].expand_as(fc)     # (1, H, W)
        x = torch.cat([fc, lead_ch], dim=0)             # (2, H, W)
        return x, self.truth[idx]


class PrecipDataModule(pl.LightningDataModule):
    """Lightning DataModule for precipitation data."""

    def __init__(
        self,
        forecast: np.ndarray,
        truth: np.ndarray,
        lead_norm: np.ndarray,
        cfg: DictConfig,
    ) -> None:
        super().__init__()
        self.forecast = forecast
        self.truth = truth
        self.lead_norm = lead_norm
        self.cfg = cfg

    def setup(self, stage: str | None = None) -> None:
        n = len(self.forecast)
        n_train = int(n * self.cfg.data.train_split)
        n_val = int(n * self.cfg.data.val_split)
        self.train_ds = PrecipitationDataset(
            self.forecast[:n_train],
            self.truth[:n_train],
            self.lead_norm[:n_train],
        )
        self.val_ds = PrecipitationDataset(
            self.forecast[n_train:n_train + n_val],
            self.truth[n_train:n_train + n_val],
            self.lead_norm[n_train:n_train + n_val],
        )
        self.test_ds = PrecipitationDataset(
            self.forecast[n_train + n_val:],
            self.truth[n_train + n_val:],
            self.lead_norm[n_train + n_val:],
        )

    def train_dataloader(self) -> DataLoader:
        return DataLoader(
            self.train_ds,
            batch_size=self.cfg.data.batch_size,
            shuffle=True,
        )

    def val_dataloader(self) -> DataLoader:
        return DataLoader(
            self.val_ds, batch_size=self.cfg.data.batch_size
        )

    def test_dataloader(self) -> DataLoader:
        return DataLoader(
            self.test_ds, batch_size=self.cfg.data.batch_size
        )


def create_datamodule(
    data: dict[str, Any],
    cfg: DictConfig,
) -> PrecipDataModule:
    """Instantiate and set up the Lightning DataModule.

    Args:
        data: Dict returned by ``load_data()``.
        cfg: Configuration with ``data.*`` split / batch keys.

    Returns:
        Ready-to-use ``PrecipDataModule`` (``setup()`` already called).
    """
    dm = PrecipDataModule(
        data["forecast"], data["truth"], data["lead_norm"], cfg
    )
    dm.setup()
    print(
        f"Train: {len(dm.train_ds)}, "
        f"Val: {len(dm.val_ds)}, "
        f"Test: {len(dm.test_ds)}"
    )
    return dm


dm = create_datamodule(data, cfg)

## 4. Model: Simple CNN for Post-Processing

A residual CNN that learns to correct the forecast. The input has **two channels**:
1. The precipitation forecast field
2. A constant field encoding the normalised lead time (0 → 1)

The model predicts a *correction field* (1 channel) that is added to the forecast channel via a residual connection.

In [ ]:
class PrecipPostProcessor(pl.LightningModule):
    """CNN that learns a lead-time-aware residual correction."""

    def __init__(
        self,
        cfg: DictConfig,
        loss_fn: nn.Module | None = None,
        optimizer_cls: type[torch.optim.Optimizer] | None = None,
        activation_cls: type[nn.Module] | None = None,
    ) -> None:
        super().__init__()
        self.save_hyperparameters(ignore=["loss_fn"])
        self.cfg = cfg
        self.loss_fn = loss_fn or nn.MSELoss()
        self.optimizer_cls = optimizer_cls or torch.optim.Adam
        activation_cls = activation_cls or nn.ReLU

        layers: list[nn.Module] = []
        in_ch: int = cfg.model.in_channels
        for i in range(cfg.model.n_layers):
            out_ch = (
                cfg.model.hidden_channels
                if i < cfg.model.n_layers - 1
                else cfg.model.out_channels
            )
            layers.append(
                nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1)
            )
            if i < cfg.model.n_layers - 1:
                layers.append(activation_cls())
            in_ch = out_ch
        self.net = nn.Sequential(*layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        correction = self.net(x)              # (B, 1, H, W)
        return x[:, :1] + correction          # residual

    def training_step(
        self,
        batch: tuple[torch.Tensor, torch.Tensor],
        batch_idx: int,
    ) -> torch.Tensor:
        x, truth = batch
        pred = self(x)
        loss = self.loss_fn(pred, truth)
        self.log("train_loss", loss, prog_bar=True)
        return loss

    def validation_step(
        self,
        batch: tuple[torch.Tensor, torch.Tensor],
        batch_idx: int,
    ) -> torch.Tensor:
        x, truth = batch
        pred = self(x)
        loss = self.loss_fn(pred, truth)
        mae = F.l1_loss(pred, truth)
        self.log("val_loss", loss, prog_bar=True)
        self.log("val_mae", mae)
        return loss

    def configure_optimizers(self) -> torch.optim.Optimizer:
        return self.optimizer_cls(
            self.parameters(), lr=self.cfg.training.learning_rate
        )


def create_model(
    cfg: DictConfig,
    model_cls: type[pl.LightningModule] = PrecipPostProcessor,
    **model_kwargs: Any,
) -> pl.LightningModule:
    """Build the CNN post-processor from config.

    Args:
        cfg: Configuration with ``model.*`` architecture keys.
        model_cls: LightningModule class to instantiate (default:
            ``PrecipPostProcessor``).
        **model_kwargs: Extra keyword arguments forwarded to
            ``model_cls.__init__`` (e.g. ``loss_fn``,
            ``optimizer_cls``, ``activation_cls``).

    Returns:
        Untrained model.
    """
    model = model_cls(cfg, **model_kwargs)
    print(model)
    return model


model = create_model(cfg)


## 5. Training with Lightning + MLFlow Tracking

Lightning's `Trainer` handles the training loop, GPU placement, and logging. We wrap the run in an **MLFlow context** so that every hyperparameter and metric is tracked automatically. The `train()` function returns both the trainer and a dict of final metrics converted back to **mm** using the normalisation scale factor.

In [ ]:
def train(
    model: pl.LightningModule,
    datamodule: PrecipDataModule,
    cfg: DictConfig,
    precip_scale: float,
) -> tuple[pl.Trainer, dict[str, float]]:
    """Train the model with Lightning and log to MLFlow.

    Args:
        model: The CNN to train.
        datamodule: Lightning DataModule (already set up).
        cfg: Configuration with ``training.*`` and ``mlflow.*`` keys.
        precip_scale: ``p99 - p1`` factor to convert normalised metrics
            back to mm.

    Returns:
        Tuple of ``(trainer, metrics)`` where *metrics* contains
        ``val_rmse_mm`` and ``val_mae_mm``.
    """
    mlflow.set_experiment(cfg.mlflow.experiment_name)

    with mlflow.start_run(run_name="cnn_postprocessor_v1"):
        mlflow.log_params(OmegaConf.to_container(cfg, resolve=True))

        trainer = pl.Trainer(
            max_epochs=cfg.training.max_epochs,
            accelerator=cfg.training.accelerator,
            enable_progress_bar=True,
            log_every_n_steps=5,
        )
        trainer.fit(model, datamodule)

        metrics: dict[str, float] = {}
        val_loss = trainer.callback_metrics.get("val_loss")
        val_mae = trainer.callback_metrics.get("val_mae")
        if val_loss is not None:
            metrics["val_rmse_mm"] = float(
                val_loss.item() ** 0.5 * precip_scale
            )
            mlflow.log_metric(
                "final_val_rmse_mm", metrics["val_rmse_mm"]
            )
        if val_mae is not None:
            metrics["val_mae_mm"] = float(
                val_mae.item() * precip_scale
            )
            mlflow.log_metric(
                "final_val_mae_mm", metrics["val_mae_mm"]
            )

    rmse = metrics.get("val_rmse_mm")
    mae = metrics.get("val_mae_mm")
    if rmse is not None:
        print(f"\nFinal val RMSE: {rmse:.4f} mm")
    if mae is not None:
        print(f"Final val MAE:  {mae:.4f} mm")

    return trainer, metrics


trainer, metrics = train(model, dm, cfg, data["precip_scale"])


## 6. Evaluation & Visualization

The `predict()` function runs inference on the test set and **denormalises** all outputs back to **mm** so that metrics are physically meaningful. `evaluate()` then produces comparison maps and per-lead-time skill curves.

In [ ]:
def predict(
    model: pl.LightningModule,
    datamodule: PrecipDataModule,
    data: dict[str, Any],
) -> dict[str, np.ndarray]:
    """Run inference on the test set and denormalise to mm.

    Args:
        model: Trained CNN.
        datamodule: DataModule with ``test_ds`` populated.
        data: Dict returned by ``load_data()`` (for normalisation stats
            and coordinates).

    Returns:
        Dict with ``forecast``, ``truth``, ``pred`` (all in mm),
        ``lead`` (normalised [0, 1]), and ``lats`` / ``lons``.
    """
    precip_p1 = data["precip_p1"]
    precip_scale = data["precip_scale"]

    def _to_mm(x: np.ndarray) -> np.ndarray:
        return x * precip_scale + precip_p1

    model.eval()
    test_ds = datamodule.test_ds
    test_x = torch.stack(
        [test_ds[i][0] for i in range(len(test_ds))]
    )

    with torch.no_grad():
        test_pred = model(test_x)

    return {
        "forecast": _to_mm(test_x[:, 0].numpy()),
        "truth": _to_mm(test_ds.truth.squeeze(1).numpy()),
        "pred": _to_mm(test_pred.squeeze(1).numpy()),
        "lead": test_ds.lead_norm.numpy(),
        "lats": data["lats"],
        "lons": data["lons"],
    }


predictions = predict(model, dm, data)


In [ ]:
def evaluate(
    predictions: dict[str, np.ndarray],
    n_steps: int,
    max_hours: float,
) -> None:
    """Plot comparison maps and per-lead-time RMSE / MAE curves.

    Args:
        predictions: Dict returned by ``predict()`` with keys
            ``forecast``, ``truth``, ``pred``, ``lead``, ``lats``,
            ``lons`` (all in mm except ``lead``).
        n_steps: Number of lead-time steps per initialisation.
        max_hours: Maximum lead time in hours (for axis labels).
    """
    fc = predictions["forecast"]
    tr = predictions["truth"]
    pp = predictions["pred"]
    lead = predictions["lead"]
    lats = predictions["lats"]
    lons = predictions["lons"]
    lon2d, lat2d = np.meshgrid(lons, lats)

    # --- Comparison maps ---
    proj = ccrs.PlateCarree()
    fig, axes = plt.subplots(
        2, 3, figsize=(18, 10), subplot_kw={"projection": proj}
    )

    for row, idx in enumerate([0, n_steps - 1]):
        lead_h = lead[idx] * max_hours
        vmin = min(tr[idx].min(), fc[idx].min())
        vmax = max(tr[idx].max(), fc[idx].max())

        panels = [
            (f"HRES Forecast ({lead_h:.0f}h)", fc[idx]),
            (f"Post-Processed ({lead_h:.0f}h)", pp[idx]),
            ("ERA5 Truth", tr[idx]),
        ]
        for col, (title, d) in enumerate(panels):
            ax = axes[row, col]
            im = ax.pcolormesh(
                lon2d, lat2d, d, cmap="Blues",
                vmin=vmin, vmax=vmax, transform=proj,
            )
            ax.add_feature(cfeature.COASTLINE, linewidth=0.5)
            ax.add_feature(
                cfeature.BORDERS, linewidth=0.3, linestyle="--"
            )
            ax.gridlines(
                draw_labels=(row == 1), linewidth=0.3, alpha=0.5
            )
            ax.set_title(title)
            plt.colorbar(
                im, ax=ax, orientation="horizontal",
                pad=0.05, shrink=0.8,
            )

    plt.suptitle(
        "Post-Processing Results — Test Set (mm)",
        fontsize=14, y=1.02,
    )
    plt.tight_layout()
    plt.show()

    # --- Per-lead-time skill curves ---
    unique_leads = np.unique(lead)
    lead_hours = unique_leads * max_hours
    raw_rmses, pp_rmses = [], []
    raw_maes, pp_maes = [], []

    for lv in unique_leads:
        mask = lead == lv
        raw_rmses.append(
            np.sqrt(np.mean((fc[mask] - tr[mask]) ** 2))
        )
        pp_rmses.append(
            np.sqrt(np.mean((pp[mask] - tr[mask]) ** 2))
        )
        raw_maes.append(np.mean(np.abs(fc[mask] - tr[mask])))
        pp_maes.append(np.mean(np.abs(pp[mask] - tr[mask])))

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

    ax1.plot(lead_hours, raw_rmses, "o-", label="Raw HRES")
    ax1.plot(lead_hours, pp_rmses, "s-", label="Post-processed (CNN)")
    ax1.set_xlabel("Lead time (hours)")
    ax1.set_ylabel("RMSE (mm)")
    ax1.set_title("RMSE vs lead time (test set)")
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    ax2.plot(lead_hours, raw_maes, "o-", label="Raw HRES")
    ax2.plot(lead_hours, pp_maes, "s-", label="Post-processed (CNN)")
    ax2.set_xlabel("Lead time (hours)")
    ax2.set_ylabel("MAE (mm)")
    ax2.set_title("MAE vs lead time (test set)")
    ax2.legend()
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

    # --- Overall numbers ---
    raw_rmse = np.sqrt(np.mean((fc - tr) ** 2))
    pp_rmse = np.sqrt(np.mean((pp - tr) ** 2))
    raw_mae = np.mean(np.abs(fc - tr))
    pp_mae = np.mean(np.abs(pp - tr))
    print(f"Raw HRES forecast RMSE: {raw_rmse:.4f} mm")
    print(f"Post-processed RMSE:    {pp_rmse:.4f} mm")
    print(
        f"RMSE improvement:       "
        f"{(1 - pp_rmse / raw_rmse) * 100:.1f}%"
    )
    print(f"\nRaw HRES forecast MAE:  {raw_mae:.4f} mm")
    print(f"Post-processed MAE:     {pp_mae:.4f} mm")
    print(
        f"MAE improvement:        "
        f"{(1 - pp_mae / raw_mae) * 100:.1f}%"
    )


evaluate(predictions, data["n_steps"], data["max_hours"])

## 7. Playground

Run the full pipeline in one cell. **Change the parameters below** and re-run to experiment!

You can customise:
- **Config values** — `max_lead_time`, `max_epochs`, `hidden_channels`, `n_layers`, `learning_rate`, `batch_size`
- **Loss function** — swap `nn.MSELoss()` for `nn.L1Loss()`, `nn.HuberLoss()`, etc.
- **Optimizer** — try `torch.optim.AdamW`, `torch.optim.SGD`, etc.
- **Activation** — try `nn.GELU`, `nn.SiLU`, `nn.LeakyReLU`, etc.
- **Model class** — define your own `LightningModule` and pass it via `model_cls`


In [ ]:
# ── 1. Config (change these!) ──────────────────────────────────
cfg = load_config(
    "../configs/precipitation.yaml",
    max_lead_time="2 days",
    max_epochs=20,
    hidden_channels=16,
    n_layers=3,
    learning_rate=1e-3,
    batch_size=32,
)

# ── 2. Loss function ──────────────────────────────────────────
# Try: nn.L1Loss(), nn.HuberLoss(), nn.SmoothL1Loss()
loss_fn = nn.MSELoss()

# ── 3. Optimizer ───────────────────────────────────────────────
# Try: torch.optim.SGD, torch.optim.AdamW
optimizer_cls = torch.optim.Adam

# ── 4. Activation ─────────────────────────────────────────────
# Try: nn.GELU, nn.SiLU, nn.LeakyReLU
activation_cls = nn.ReLU

# ── 5. Run pipeline ───────────────────────────────────────────
data = load_data(cfg)
plot_raw_data(data)
dm = create_datamodule(data, cfg)
model = create_model(
    cfg,
    loss_fn=loss_fn,
    optimizer_cls=optimizer_cls,
    activation_cls=activation_cls,
)
trainer, metrics = train(model, dm, cfg, data["precip_scale"])
predictions = predict(model, dm, data)
evaluate(predictions, data["n_steps"], data["max_hours"])


## 8. Exercises

Use the **Playground** cell above to experiment — change one parameter at a time and re-run.

1. **Increase the horizon**: Set `max_lead_time="5 days"` or `"10 days"` — how does the per-lead-time skill curve change?
2. **Change the architecture**: Try `n_layers=5` or `hidden_channels=32`. What happens with a deeper / wider model?
3. **Compare optimizers**: Set `optimizer_cls = torch.optim.SGD` or `torch.optim.AdamW` in the playground — how does convergence differ?
4. **Add more input channels**: Load additional HRES variables from the Zarr store (e.g., `2m_temperature`, `geopotential`) as extra CNN input channels
5. **Try a different region**: Change `lat_slice` to select different latitude bands and see how precipitation patterns differ
6. **Check MLFlow**: Run `!mlflow ui` and open the link to explore logged experiments
